In [1]:
!pip install scikit-learn
!pip install nltk
!pip install emoji

In [2]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
import emoji

nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to /home/julyanna/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/julyanna/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/julyanna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
rappler_docs = pd.read_excel('rappler.xlsx')

In [4]:
rappler_docs

,Unnamed: 0,Title,Url,Publish Date,Content,Related Topics
0,0,SC denies Alice Guo's challenge of Senate subp...,https://www.rappler.com/newsbreak/explainers/s...,2025-08-16 02:00:00,"SUMMARY This is AI generated summarization, wh...","Alice Guo,Congress of the Philippines,Philippi..."
1,1,WATCH: House reactivates quad committee that p...,https://www.rappler.com/philippines/video-hous...,2025-08-05 12:44:14,"MANILA, Philippines – The House of Representat...","Duterte administration,extrajudicial killings,..."
2,2,5 things to watch out for in the 20th Congress,https://www.rappler.com/philippines/20th-congr...,2025-07-27 09:40:49,"SUMMARY This is AI generated summarization, wh...","#ScamAlert,Budget Watch,Chiz Escudero,Ferdinan..."
3,3,WATCH: Saan dumadaan ang perang na-scam?,https://www.rappler.com/newsbreak/podcasts-vid...,2025-07-25 05:42:54,"MANILA, Philippines – Ang perang nasa-scam ay ...","cryptocurrency,gambling,POGOs"
4,4,[ANALYSIS] Why the Konektadong Pinoy Act shoul...,https://www.rappler.com/business/opinion-why-k...,2025-07-23 05:20:49,"SUMMARY This is AI generated summarization, wh...","Congress of the Philippines,DICT,Internet infr..."
...,...,...,...,...,...,...
245,245,Metro Manila under state of calamity | The wRap,https://www.rappler.com/video/daily-wrap/july-...,2024-07-24 14:46:53,Here are today’s headlines – the latest news i...,"Hollywood celebrities,POGOs,Sara Duterte,Unite..."
246,246,Philippines orders foreign workers in offshore...,https://www.rappler.com/philippines/foreign-wo...,2024-07-24 06:34:17,"SUMMARY This is AI generated summarization, wh...",NaN
247,247,DOLE to help find jobs for displaced POGO workers,https://www.rappler.com/business/dole-help-fin...,2024-07-24 05:58:57,"SUMMARY This is AI generated summarization, wh...","Philippine labor,POGOs"
248,248,Marcos’ POGO ban welcomed by Filipinos | The wRap,https://www.rappler.com/video/daily-wrap/july-...,2024-07-23 14:16:00,Here are today’s headlines – the latest news i...,"Ferdinand Marcos Jr.,Kamala Harris,Martin Romu..."


In [5]:
from pandas.errors import EmptyDataError

try:
  basic_stopwords = list(
    # Handle empty data error
    pd.read_csv('basic_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  basic_stopwords = []

try:
  domain_stopwords = list(
    pd.read_csv('domain_stopwords.txt', header=None).values.flatten()
  )
except EmptyDataError:
  domain_stopwords = []

In [6]:
def preprocess_text(corpus, text_column='text'):
  cleaned_corpus = corpus.copy()

  # Lowercase
  cleaned_corpus['cleaned_text'] = cleaned_corpus[text_column].str.lower()

  # Lemmatize (by default, lemmatize nouns)
  # Other options:
  #   'v' for verbs
  #   'a' for adjectives
  #   'r' for adverbs
  #   's' for satellites adjectives (adjectives that appear after verbs)
  lemmatizer = WordNetLemmatizer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [lemmatizer.lemmatize(word, pos='n') for word in text.split()]
      )
  )

  # Stemmer
  stemmer = PorterStemmer()
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
      lambda text: ' '.join(
        [stemmer.stem(word) for word in text.split()]
      )
  )

  # Remove NLTK stopwords
  en_stopwords_list = stopwords.words('english')
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [
        word for word in text.split() if word not in en_stopwords_list
      ]
    )
  )

  # Remove basic stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in basic_stopwords]
    )
  )

  # Remove domain stopwords
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in domain_stopwords]
    )
  )

  # Remove trailing and leading whitespaces
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.strip()

  # Remove non-alphanumeric characters
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\W', ' ', regex=True)

  # Remove numbers
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].str.replace(r'\d+', ' ', regex=True)

  # Remove emojis using emoji library
  cleaned_corpus['cleaned_text'] = cleaned_corpus['cleaned_text'].apply(
    lambda text: ' '.join(
      [word for word in text.split() if word not in list(emoji.EMOJI_DATA.keys())]
    )
  )

  return cleaned_corpus['cleaned_text']

In [9]:
rappler_docs['cleaned_text'] = preprocess_text(rappler_docs, text_column="Content")

In [10]:
rappler_docs.to_excel("rappler.xlsx", index=False)

In [11]:
rappler_docs

,Unnamed: 0,Title,Url,Publish Date,Content,Related Topics,cleaned_text
0,0,SC denies Alice Guo's challenge of Senate subp...,https://www.rappler.com/newsbreak/explainers/s...,2025-08-16 02:00:00,"SUMMARY This is AI generated summarization, wh...","Alice Guo,Congress of the Philippines,Philippi...",summari thi ai gener summarization may errors ...
1,1,WATCH: House reactivates quad committee that p...,https://www.rappler.com/philippines/video-hous...,2025-08-05 12:44:14,"MANILA, Philippines – The House of Representat...","Duterte administration,extrajudicial killings,...",manila philippin hous repres approv reconstitu...
2,2,5 things to watch out for in the 20th Congress,https://www.rappler.com/philippines/20th-congr...,2025-07-27 09:40:49,"SUMMARY This is AI generated summarization, wh...","#ScamAlert,Budget Watch,Chiz Escudero,Ferdinan...",summari thi ai gener summarization may errors ...
3,3,WATCH: Saan dumadaan ang perang na-scam?,https://www.rappler.com/newsbreak/podcasts-vid...,2025-07-25 05:42:54,"MANILA, Philippines – Ang perang nasa-scam ay ...","cryptocurrency,gambling,POGOs",manila philippin ang perang nasa scam ay dumad...
4,4,[ANALYSIS] Why the Konektadong Pinoy Act shoul...,https://www.rappler.com/business/opinion-why-k...,2025-07-23 05:20:49,"SUMMARY This is AI generated summarization, wh...","Congress of the Philippines,DICT,Internet infr...",summari thi ai gener summarization may errors ...
...,...,...,...,...,...,...,...
245,245,Metro Manila under state of calamity | The wRap,https://www.rappler.com/video/daily-wrap/july-...,2024-07-24 14:46:53,Here are today’s headlines – the latest news i...,"Hollywood celebrities,POGOs,Sara Duterte,Unite...",today headlin latest news philippin around wor...
246,246,Philippines orders foreign workers in offshore...,https://www.rappler.com/philippines/foreign-wo...,2024-07-24 06:34:17,"SUMMARY This is AI generated summarization, wh...",NaN,summari thi ai gener summarization may errors ...
247,247,DOLE to help find jobs for displaced POGO workers,https://www.rappler.com/business/dole-help-fin...,2024-07-24 05:58:57,"SUMMARY This is AI generated summarization, wh...","Philippine labor,POGOs",summari thi ai gener summarization may errors ...
248,248,Marcos’ POGO ban welcomed by Filipinos | The wRap,https://www.rappler.com/video/daily-wrap/july-...,2024-07-23 14:16:00,Here are today’s headlines – the latest news i...,"Ferdinand Marcos Jr.,Kamala Harris,Martin Romu...",today headlin latest news philippin around wor...


In [14]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import pyLDAvis
import numpy as np

In [16]:
vectorizer = CountVectorizer(
  max_df=0.95,  # terms that appears 95% within the corpus
  min_df=2,  # terms that appear at 2x within the corpus
  stop_words='english'  # ignore english stopwords
)
doc_term_matrix = vectorizer.fit_transform(
  rappler_docs['cleaned_text']
)

doc_term_matrix.toarray().shape

(250, 5387)

In [17]:
pd.DataFrame(
  doc_term_matrix.toarray()
)

,0,1,2,3,4,5,6,7,8,9,...,5377,5378,5379,5380,5381,5382,5383,5384,5385,5386
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,2,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
246,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
247,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
248,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
LDA = LatentDirichletAllocation(
  n_components=5,  # no. of topics
  random_state=42  # random seed for replicability
)
LDA.fit(doc_term_matrix)

LatentDirichletAllocation(n_components=5, random_state=42)

In [19]:
len(LDA.components_)

5

In [20]:
topic_term_matrix = LDA.components_
pd.DataFrame(topic_term_matrix)

,0,1,2,3,4,5,6,7,8,9,...,5377,5378,5379,5380,5381,5382,5383,5384,5385,5386
0,0.200000,0.200000,4.2,0.200723,0.201092,1.407160,1.185107,0.20029,0.208191,0.206154,...,8.178257,0.201309,1.860694,0.201716,0.200000,0.201471,4.080457,0.814251,0.222722,1.316446
1,0.200000,0.200156,0.2,0.236550,0.206690,0.200000,1.200000,0.20000,11.587191,12.157214,...,0.219364,0.200145,0.202879,0.201429,1.131814,0.202295,1.208947,0.200000,5.010115,9.828261
2,1.421886,3.199844,0.2,28.161275,8.762791,0.201862,0.200000,2.19971,0.206615,0.230712,...,0.200585,0.201716,8.853971,11.354825,0.200000,0.200282,0.476810,0.330135,0.209029,0.799206
3,0.975435,0.200000,0.2,0.201452,1.628446,3.613263,0.200000,0.20000,0.201911,0.201980,...,0.200852,3.196067,21.303524,22.225731,5.268088,0.200000,18.471711,1.428547,1.337057,9.702400
4,0.202678,0.200000,0.2,0.200000,0.200981,2.577714,0.214892,0.20000,0.796092,0.203941,...,0.200942,0.200763,1.778932,8.016299,0.200097,3.195952,45.762075,3.227067,0.221078,64.353687


In [21]:
topic1 = topic_term_matrix[0]
topic1

# sort the scores from lowest to highest
# will return all terms
topic1.argsort()

# select top 10 terms within topic1
topic1_term_index = topic1.argsort()[-10:]

# convert back to words/terms
[vectorizer.get_feature_names_out()[index] for index in topic1_term_index]

['duterte',
 'drug',
 'report',
 'presid',
 'ha',
 'pogo',
 'said',
 'hi',
 'wa',
 'yang']

In [22]:
for topic_number, topic in enumerate(LDA.components_):
  print(f'The top 10 words for topic #{topic_number}')
  print(
    [
      vectorizer.get_feature_names_out()[term_index] for term_index in topic.argsort()[-5:]
    ]
  )
  print("\n")

The top 10 words for topic #0
['pogo', 'said', 'hi', 'wa', 'yang']


The top 10 words for topic #1
['senat', 'hous', 'ong', 'wa', 'guo']


The top 10 words for topic #2
['mga', 'ng', 'ang', 'sa', 'na']


The top 10 words for topic #3
['pogo', 'oper', 'guo', 'wa', 'said']


The top 10 words for topic #4
['oper', 'game', 'wa', 'said', 'pogo']




In [23]:
doc_topic_matrix = LDA.transform(doc_term_matrix)
pd.DataFrame(doc_topic_matrix)

,0,1,2,3,4
0,0.000437,0.998253,0.000433,0.000439,0.000438
1,0.001749,0.993078,0.001716,0.001728,0.001728
2,0.516861,0.481911,0.000408,0.000410,0.000410
3,0.000329,0.000330,0.998683,0.000329,0.000329
4,0.835031,0.000448,0.000442,0.000448,0.163631
...,...,...,...,...,...
245,0.652109,0.231888,0.002238,0.002269,0.111497
246,0.269318,0.001538,0.001527,0.383713,0.343903
247,0.000776,0.000771,0.045238,0.000774,0.952440
248,0.651460,0.341251,0.002399,0.002422,0.002467


In [26]:
rappler_docs['Topic'] = doc_topic_matrix.argmax(axis=1)
rappler_docs[['cleaned_text', 'Topic']]
rappler_docs.to_excel('topic_documents.xlsx')

In [28]:
vocab = vectorizer.get_feature_names_out()
doc_lengths = [
  len(doc) for doc in rappler_docs['cleaned_text']
]
term_freq = doc_term_matrix.sum(axis=0)
term_freqs = np.array(term_freq).flatten()
vis_data = pyLDAvis.prepare(topic_term_matrix, doc_topic_matrix, doc_lengths, vocab, term_freqs)

pyLDAvis.display(vis_data)

TypeError: DataFrame.drop() takes from 1 to 2 positional arguments but 3 were given